# 03 — Klasik Makine Öğrenmesi

RF · GBM · SVM · XGBoost · LightGBM — Leave-One-Driver-Out (LODO) çapraz doğrulama.

Her fold'da:
- NaN → eğitim sütun ortalamasıyla doldurulur
- StandardScaler **yalnızca eğitim verisine** fit edilir (veri sızıntısı yok)
- Sonuçlar `results/metrics/metrics.json` ve `results/figures/` dizinlerine kaydedilir

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from models import (
    build_random_forest,
    build_xgboost, build_lightgbm,
)
from features import get_feature_columns
from evaluate import plot_feature_importance
from experiment import compare_models

PROC_DIR    = Path('../data/processed')
RESULTS_DIR = Path('../results')
FIG_DIR     = RESULTS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / 'models').mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_parquet(PROC_DIR / 'features_5s_2s.parquet')
print(f'Tablo boyutu : {df.shape}')
print(f'Sürücüler    : {sorted(df["driver"].unique())}')
print()
print(df['behavior'].value_counts().to_string())

Tablo boyutu : (10434, 151)
Sürücüler    : ['D1', 'D2', 'D3', 'D4', 'D5', 'D6']

behavior
normal        4418
economic      3348
aggressive    2668


In [3]:
feat_cols = get_feature_columns(df)
print(f'Toplam özellik : {len(feat_cols)}')

# NaN içeren satırları düşür (varsa)
df_clean = df.dropna(subset=feat_cols).reset_index(drop=True)
print(f'Temiz pencere  : {len(df_clean)}')
print(f'Sınıf dağılımı : {dict(pd.Series(df_clean["label"]).value_counts().sort_index())}')

Toplam özellik : 145
Temiz pencere  : 10434
Sınıf dağılımı : {0: np.int64(4418), 1: np.int64(2668), 2: np.int64(3348)}


## Tüm Modeller — LODO Değerlendirmesi

`compare_models()` her model için 6 fold çalıştırır:
- per-fold StandardScaler (sadece train'e fit)
- Eğitilen her fold modeli `results/models/` dizinine kaydedilir
- Metrikler `results/metrics/metrics.json` dosyasına biriktirilir

In [ ]:
model_builders = {
    'RandomForest' : build_random_forest,
    'XGBoost'      : build_xgboost,
    'LightGBM'     : build_lightgbm,
}

all_results, comp_df = compare_models(
    df=df_clean,
    model_builders=model_builders,
    feature_cols=feat_cols,
    results_dir=RESULTS_DIR,
    verbose=True,
)

## Karşılaştırma Tablosu

In [ ]:
# Tablo: Acc mean±std, F1 mean±std, per-class F1
display_cols = ['Model', 'Acc mean', 'Acc std', 'F1 mean', 'F1 std']
per_class_cols = [c for c in comp_df.columns if c.startswith('F1_')]
print(comp_df[display_cols + per_class_cols].to_string(index=False, float_format='{:.3f}'.format))

In [ ]:
# Grafik results/figures/model_comparison.png olarak zaten kaydedildi
img_path = FIG_DIR / 'model_comparison.png'
if img_path.exists():
    from IPython.display import Image
    display(Image(filename=str(img_path)))

## Fold Bazlı Detay — En İyi Model

In [ ]:
best_name = comp_df.iloc[0]['Model']
print(f'En iyi model: {best_name}\n')

fold_rows = []
for m in all_results[best_name]['fold_metrics']:
    fold_rows.append({
        'Fold'  : m['fold'] + 1,
        'Test'  : m['test_driver'],
        'Acc'   : round(m['accuracy'], 3),
        'F1'    : round(m['macro_f1'], 3),
        'ROC-AUC': round(m.get('roc_auc_macro') or 0, 3),
    })

fold_df = pd.DataFrame(fold_rows)
print(fold_df.to_string(index=False))

print(f'\nOrtalama  Acc={np.mean(fold_df["Acc"]):.3f}  F1={np.mean(fold_df["F1"]):.3f}')

In [ ]:
# Karışıklık matrisi (kaydedilmiş görsel)
cm_path = FIG_DIR / f'cm_{best_name.lower().replace(" ", "_")}.png'
if cm_path.exists():
    from IPython.display import Image
    display(Image(filename=str(cm_path)))

## Özellik Önemi — Ağaç Tabanlı Modeller

In [ ]:
from models import load_model
models_dir = RESULTS_DIR / 'models'

tree_models = ['RandomForest', 'XGBoost', 'LightGBM']

for mname in tree_models:
    if mname not in all_results:
        continue
    n_folds = all_results[mname]['n_folds']
    last_fold_idx = n_folds - 1
    last_test_driver = all_results[mname]['fold_metrics'][last_fold_idx]['test_driver']
    model_path = models_dir / f'{mname}_fold{last_fold_idx}_test{last_test_driver}.joblib'

    if not model_path.exists():
        continue

    model = load_model(model_path)
    if not hasattr(model, 'feature_importances_'):
        continue

    fig = plot_feature_importance(
        feat_cols,
        model.feature_importances_,
        top_n=25,
        title=f'{mname} — En Önemli 25 Özellik (Son Fold)',
        save_path=FIG_DIR / f'feature_importance_{mname.lower()}.png',
    )
    plt.show()

## Sınıf Dağılımı

In [ ]:
from evaluate import plot_class_distribution
plot_class_distribution(
    df_clean['label'].values,
    title='Etiket Dağılımı (tüm veri)',
    save_path=FIG_DIR / 'class_distribution.png',
)
plt.show()